<a href="https://colab.research.google.com/github/qiyuriko/ICT304-Assignment1/blob/main/ICT304_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
!unzip -q "/content/bird dataset.zip" -d /content/bird_data
print("Unzip completed. Contents extracted to /content/bird_data")

Unzip completed. Contents extracted to /content/bird_data


In [5]:
import os
from collections import Counter

# The dataset unzips into "/content/bird_data/bird dataset"
base_path = "/content/bird_data/bird dataset"

if os.path.exists(base_path):
    print(f"--- Exploring Dataset Classes in {base_path} ---")
    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)

        if os.path.isdir(folder_path):
            files = os.listdir(folder_path)
            extensions = Counter(os.path.splitext(f)[1].lower() for f in files)

            print(f"\nClass: {folder}")
            print("Total files:", len(files))
            print("Formats:", extensions)
else:
    print(f"Directory not found: {base_path}. Please make sure the dataset is extracted successfully.")

--- Exploring Dataset Classes in /content/bird_data/bird dataset ---

Class: marsh sandpiper
Total files: 140
Formats: Counter({'.jpg': 140})

Class: hooded merganser
Total files: 60
Formats: Counter({'.jpg': 60})

Class: common kingfisher
Total files: 85
Formats: Counter({'.jpg': 79, '.png': 6})

Class: greater cormorant
Total files: 140
Formats: Counter({'.jpg': 140})

Class: mallard
Total files: 60
Formats: Counter({'.jpg': 60})

Class: black crowned night heron
Total files: 156
Formats: Counter({'.jpg': 140, '.png': 16})


In [6]:
import os

def list_dir_structure(startpath, max_depth=3):
    for root, dirs, files in os.walk(startpath):
        depth = root.replace(startpath, '').count(os.sep)
        if depth < max_depth:
            indent = ' ' * 4 * (depth)
            print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')

list_dir_structure('/content/bird_data')

bird_data/ (0 files)
    bird dataset/ (0 files)
        marsh sandpiper/ (140 files)
        hooded merganser/ (60 files)
        common kingfisher/ (85 files)
        greater cormorant/ (140 files)
        mallard/ (60 files)
        black crowned night heron/ (156 files)


In [7]:
import shutil
import os

# Remove existing directory if it exists
bird_data_path = '/content/bird_data'
if os.path.exists(bird_data_path):
    shutil.rmtree(bird_data_path)
    print(f"Removed {bird_data_path} and all its contents.")

# Recreate clean folder
os.makedirs(bird_data_path, exist_ok=True)

# Unzip the uploaded dataset into it
!unzip -q "/content/bird dataset.zip" -d /content/bird_data
print("Re-extraction completed successfully!")

Removed /content/bird_data and all its contents.
Re-extraction completed successfully!


In [8]:
# Verify the cleaned and newly extracted folder structure
list_dir_structure('/content/bird_data')

bird_data/ (0 files)
    bird dataset/ (0 files)
        marsh sandpiper/ (140 files)
        hooded merganser/ (60 files)
        common kingfisher/ (85 files)
        greater cormorant/ (140 files)
        mallard/ (60 files)
        black crowned night heron/ (156 files)


In [9]:
import os
from PIL import Image
from collections import Counter

input_base = "/content/bird_data/bird dataset"
output_base = "/content/bird_data/preprocessed/color"
target_size = 224

def to_square(img, size=224, resample=Image.BICUBIC):
    w, h = img.size
    scale = size / max(w, h)
    img = img.resize((round(w * scale), round(h * scale)), resample)
    canvas = Image.new("RGB", (size, size), (0, 0, 0))
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas

# Preprocess the 6 species
for bird_class in os.listdir(input_base):
    input_folder = os.path.join(input_base, bird_class)
    if not os.path.isdir(input_folder):
        continue

    output_folder = os.path.join(output_base, bird_class)
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):
        input_file = os.path.join(input_folder, file)
        output_file = os.path.join(output_folder, file)

        try:
            with Image.open(input_file) as img:
                img = img.convert("RGB")
                img = to_square(img, target_size)
                img.save(output_file)
        except Exception as e:
            print(f"Error processing {input_file}: {e}")

print("Preprocessing completed successfully for 6 species!")

Preprocessing completed successfully for 6 species!


In [10]:
# Explore preprocessed classes and counts
print("Preprocessed Class Counts:")
for c in sorted(os.listdir(output_base)):
    path = os.path.join(output_base, c)
    if os.path.isdir(path):
        print(f"{c}: {len(os.listdir(path))} images")

Preprocessed Class Counts:
black crowned night heron: 156 images
common kingfisher: 85 images
greater cormorant: 140 images
hooded merganser: 60 images
mallard: 60 images
marsh sandpiper: 140 images


In [11]:
# Check for corrupted images in the preprocessed folder
from PIL import Image
import os

base_path = "/content/bird_data/preprocessed/color"
corrupted = []

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)
            try:
                with Image.open(file_path) as img:
                    img.verify()
            except Exception:
                corrupted.append(file_path)

print(f"Total corrupted images found: {len(corrupted)}")

Total corrupted images found: 0


In [12]:
# Explore preprocessed image sizes and color modes
from collections import Counter

base_path = "/content/bird_data/preprocessed/color"
sizes = []
modes = []

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)
            with Image.open(file_path) as img:
                sizes.append(img.size)
                modes.append(img.mode)

print("Image sizes:", Counter(sizes))
print("Color modes:", Counter(modes))

Image sizes: Counter({(224, 224): 641})
Color modes: Counter({'RGB': 641})


In [16]:
import os
import shutil
import random

# Set random seed for reproducibility
random.seed(42)

preprocessed_base = "/content/bird_data/preprocessed/color"
train_dir = "/content/bird_data/split/train"
val_dir = "/content/bird_data/split/val"
test_dir = "/content/bird_data/split/test"

# Create target directories
for d in [train_dir, val_dir, test_dir]:
    os.makedirs(d, exist_ok=True)

# Split ratio: 80% train, 10% val, 10% test
for bird_class in sorted(os.listdir(preprocessed_base)):
    class_path = os.path.join(preprocessed_base, bird_class)
    if not os.path.isdir(class_path):
        continue

    # Ensure class subfolders exist in split destinations
    os.makedirs(os.path.join(train_dir, bird_class), exist_ok=True)
    os.makedirs(os.path.join(val_dir, bird_class), exist_ok=True)
    os.makedirs(os.path.join(test_dir, bird_class), exist_ok=True)

    # Get all images for this class
    images = [img for img in os.listdir(class_path)]
    random.shuffle(images)

    # Compute split boundaries
    total_imgs = len(images)
    train_idx = int(total_imgs * 0.8)
    val_idx = train_idx + int(total_imgs * 0.1)

    train_imgs = images[:train_idx]
    val_imgs = images[train_idx:val_idx]
    test_imgs = images[val_idx:]

    # Copy files to respective split directories
    for img in train_imgs:
        shutil.copy(os.path.join(class_path, img), os.path.join(train_dir, bird_class, img))
    for img in val_imgs:
        shutil.copy(os.path.join(class_path, img), os.path.join(val_dir, bird_class, img))
    for img in test_imgs:
        shutil.copy(os.path.join(class_path, img), os.path.join(test_dir, bird_class, img))

print("Dataset splitting completed successfully!")

Dataset splitting completed successfully!


In [17]:
# Verify the split distributions
print("--- Dataset Split Summary ---")
for split_name, split_path in [("Train", train_dir), ("Validation", val_dir), ("Test", test_dir)]:
    print(f"\n{split_name} Set:")
    total_split_count = 0
    for bird_class in sorted(os.listdir(split_path)):
        class_path = os.path.join(split_path, bird_class)
        if os.path.isdir(class_path):
            count = len(os.listdir(class_path))
            total_split_count += count
            print(f"  {bird_class}: {count} images")
    print(f"Total {split_name} Images: {total_split_count}")

--- Dataset Split Summary ---

Train Set:
  black crowned night heron: 124 images
  common kingfisher: 68 images
  greater cormorant: 112 images
  hooded merganser: 48 images
  mallard: 48 images
  marsh sandpiper: 112 images
Total Train Images: 512

Validation Set:
  black crowned night heron: 15 images
  common kingfisher: 8 images
  greater cormorant: 14 images
  hooded merganser: 6 images
  mallard: 6 images
  marsh sandpiper: 14 images
Total Validation Images: 63

Test Set:
  black crowned night heron: 17 images
  common kingfisher: 9 images
  greater cormorant: 14 images
  hooded merganser: 6 images
  mallard: 6 images
  marsh sandpiper: 14 images
Total Test Images: 66
